<h1>Important</h1>

- The following notebook has only personal learning purposes with no further intention. This was developed using AI tools combined with multiple iterations to refine the code given at first it does generate many errors, from documentation error and more.
- The key intention of this notebook is to show to use a model available from Hugging Face in combination with techniques from the same library to do fine-tuning of the model and show how it works.
- This is not a comercial or industry code to be used, it is just a personal academic learning of how to use different python libraries with ideas of Reinforcement Learning and LLMs.

Key considerations when running this:
- The machine that was used had installed cuda nvidia with 8 GB of capacity, and the idea was to constraint the dataset size and memory usage when training the model.
- Do not load more than 1 model if the CUDA capacity is small because it will lead to potential crushing.
- Try to use cuda and not cpu because is much faster when running.

_____

# DoRA Supervised Fine-Tuning

In [1]:
# Mathematical Foundation: W = m × (W₀ + BA) / ||W₀ + BA||
# Innovation: Weight-Decomposed LoRA separating magnitude and direction learning
# Advantage: Enhanced expressiveness over LoRA with minimal parameter overhead

"""
THEORETICAL FOUNDATION

Weight-Decomposed LoRA (DoRA) represents a significant advancement over standard LoRA
by separating magnitude and direction components of weight updates.

1. MATHEMATICAL INNOVATION:
   - Standard LoRA: W = W₀ + BA
   - DoRA decomposition: W = m × V/||V|| where V = W₀ + BA
   - Magnitude vector: m ∈ ℝ^d (learned independently)
   - Direction matrix: V = W₀ + BA (low-rank adapted)
   - Normalization: ||V|| ensures unit direction vector

2. THEORETICAL ADVANTAGE:
   - Separates scaling (magnitude) from pattern (direction) learning
   - More expressive than LoRA with only ~0.01% parameter overhead
   - Better captures the behavior of full fine-tuning
   - Enhanced learning dynamics through independent magnitude control

3. COMPUTATIONAL BENEFITS:
   - Minimal additional parameters: only d magnitude values per layer
   - Same memory efficiency as LoRA during training
   - Improved convergence properties due to decomposition
   - Better adaptation to target domains

4. IMPLEMENTATION DETAILS:
   - Magnitude vectors initialized to ||W₀ + BA||
   - Direction updates through standard LoRA mechanism
   - Forward pass: h = m ⊙ (W₀ + BA)x / ||W₀ + BA||
   - Joint optimization of magnitude and directional components

5. EMPIRICAL ADVANTAGES:
   - Superior performance on reasoning tasks
   - Better few-shot learning capabilities
   - More stable training dynamics
   - Enhanced task-specific adaptation

This implementation demonstrates DoRA's enhanced capacity while maintaining
the same computational efficiency as LoRA.
"""

import warnings

warnings.filterwarnings("ignore")

import subprocess
import sys
import torch
import gc
import os
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
import json
from datetime import datetime

# Global Parameters - Consistent across methods for comparison
MODEL_NAME = "Qwen/Qwen2-0.5B-Instruct"
TEMPERATURE = 0.1
MAX_LENGTH = 1024
MAX_NEW_TOKENS = 1024
LEARNING_RATE_PEFT = 2e-4  # Slightly adjusted for DoRA stability
NUM_TRAIN_EPOCHS = 20  # More epochs to leverage DoRA's enhanced capacity
PER_DEVICE_TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 3
WARMUP_RATIO = 0.1
LOGGING_STEPS = 4
device = "cuda" if torch.cuda.is_available() else "cpu"

# Standard evaluation questions for consistency
STANDARD_TEST_QUESTIONS = [
    "How do I cook perfect pasta?",
    "What's the secret to fluffy pancakes?",
    "How can I make my cookies soft and chewy?",
    "My bread never rises properly. Help!",
    "How do I prevent my cakes from being dry?",
]


def install_packages():
    """Install required packages for DoRA fine-tuning"""
    packages = [
        "torch",
        "transformers>=4.35.0",
        "trl>=0.7.0",
        "peft>=0.6.0",
        "datasets",
        "bitsandbytes",
        "accelerate",
    ]
    for pkg in packages:
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])
        except:
            pass


def cuda_usage():
    """Monitor CUDA memory usage for 8GB VRAM management"""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated(device) / (1024**3)
        reserved = torch.cuda.memory_reserved(device) / (1024**3)
        print(f"GPU Memory: {allocated:.2f}GB allocated, {reserved:.2f}GB reserved")
        print(f"Available: {8.0 - reserved:.2f}GB remaining")
    else:
        print("CUDA not available - using CPU")


def cleanup_memory():
    """Comprehensive memory cleanup for CUDA management"""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()


def simple_chat_test(
    model,
    tokenizer,
    prompt,
    temperature=TEMPERATURE,
    max_length=MAX_LENGTH,
    max_new_tokens=MAX_NEW_TOKENS,
):
    """Generate complete model response for evaluation"""
    model.eval()

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=max_length,
        padding=False,
    )

    if torch.cuda.is_available():
        inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        with (
            torch.autocast(device_type="cuda", dtype=torch.bfloat16)
            if torch.cuda.is_available()
            else torch.no_grad()
        ):
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=temperature,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
                repetition_penalty=1.1,
            )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True
    )
    return response.strip()


def test_model_comprehensive(model, tokenizer, model_name):
    """Comprehensive model evaluation with consistent formatting"""
    qa_results = {}

    print(f"\n" + "=" * 80)
    print(f"MODEL EVALUATION: {model_name}")
    print(f"=" * 80)

    for i, question in enumerate(STANDARD_TEST_QUESTIONS, 1):
        print(f"\nQuestion {i}/{len(STANDARD_TEST_QUESTIONS)}: {question}")
        print("-" * 60)

        response = simple_chat_test(model, tokenizer, question)
        qa_results[question] = response

        print(f"Response:\n{response}")
        print("-" * 60)

    return qa_results


def compare_model_performance(base_results, trained_results, method_name):
    """Side-by-side comparison of base vs trained model outputs"""
    print(f"\n" + "=" * 80)
    print(f"COMPARATIVE ANALYSIS: Base Model vs {method_name}")
    print(f"=" * 80)
    print("This comparison shows DoRA's enhanced adaptation capabilities")
    print("Look for improvements in nuance, detail, and response quality")
    print("=" * 80)

    for i, question in enumerate(STANDARD_TEST_QUESTIONS, 1):
        print(f"\n[QUESTION {i}]: {question}")
        print("=" * 80)

        print(f"\n[BASE MODEL OUTPUT]:")
        print("-" * 40)
        print(f"{base_results[question]}")

        print(f"\n[{method_name.upper()} MODEL OUTPUT]:")
        print("-" * 40)
        print(f"{trained_results[question]}")

        print("\n" + "=" * 80)


def create_dora_config():
    """
    DoRA Configuration - Mathematical Foundation:

    DoRA enhances LoRA by decomposing weights into magnitude and direction:
    - W = m × V/||V|| where V = W₀ + BA
    - Magnitude vector: m ∈ ℝ^d controls scaling
    - Direction matrix: V uses LoRA decomposition
    - Normalization: Ensures unit direction vector

    Benefits over LoRA:
    - Independent magnitude and direction learning
    - Better approximation of full fine-tuning behavior
    - Enhanced expressiveness with minimal overhead
    - Improved convergence properties

    Returns:
        DoRA configuration object for PEFT
    """
    return LoraConfig(
        r=16,  # Rank: Same as LoRA for fair comparison
        lora_alpha=32,  # Scaling factor for LoRA component
        target_modules=["q_proj", "v_proj"],  # Query and Value projections
        lora_dropout=0.05,  # Regularization for generalization
        bias="none",  # Focus on weight decomposition
        task_type="CAUSAL_LM",
        use_dora=True,  # Enable Weight-Decomposed LoRA
    )


def analyze_dora_parameters(model):
    """
    Analyze DoRA's parameter structure and efficiency

    DoRA Parameter Breakdown:
    - LoRA parameters: B and A matrices (directional updates)
    - Magnitude parameters: m vectors (scaling factors)
    - Total overhead: minimal increase over LoRA
    """
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())

    # Separate LoRA and magnitude parameters
    lora_params = 0
    magnitude_params = 0

    for name, param in model.named_parameters():
        if param.requires_grad:
            if "lora_magnitude_vector" in name:
                magnitude_params += param.numel()
            else:
                lora_params += param.numel()

    efficiency_ratio = trainable_params / total_params * 100
    reduction_factor = total_params // trainable_params
    magnitude_overhead = magnitude_params / lora_params * 100 if lora_params > 0 else 0

    print(f"\n" + "=" * 70)
    print("DORA PARAMETER ANALYSIS")
    print("=" * 70)
    print("Mathematical Foundation: W = m × (W₀ + BA) / ||W₀ + BA||")
    print()
    print("Parameter Breakdown:")
    print(f"  • Total parameters: {total_params:,}")
    print(f"  • LoRA parameters (B,A): {lora_params:,}")
    print(f"  • Magnitude parameters (m): {magnitude_params:,}")
    print(f"  • DoRA total trainable: {trainable_params:,}")
    print(f"  • Efficiency ratio: {efficiency_ratio:.3f}% trainable")
    print(f"  • Parameter reduction: ~{reduction_factor}x")
    print()
    print("DoRA vs LoRA Comparison:")
    print(f"  • Magnitude overhead: +{magnitude_params:,} parameters")
    print(f"  • Relative overhead: +{magnitude_overhead:.2f}% over LoRA")
    print(f"  • Enhanced expressiveness: Independent magnitude control")
    print(f"  • Memory efficiency: Comparable to LoRA")
    print()
    print("Theoretical Advantages:")
    print("  • Separates scaling from directional adaptation")
    print("  • Better approximates full fine-tuning behavior")
    print("  • Enhanced learning dynamics through decomposition")
    print("=" * 70)

    return {
        "total_params": total_params,
        "trainable_params": trainable_params,
        "lora_params": lora_params,
        "magnitude_params": magnitude_params,
        "efficiency_ratio": efficiency_ratio,
        "reduction_factor": reduction_factor,
    }


def create_cooking_dataset():
    """
    Create identical dataset to LoRA for direct comparison

    This ensures performance differences are due to DoRA's enhanced
    architecture rather than dataset variations.
    """
    return [
        {
            "text": "How do I cook perfect pasta?\n\nHere's the foolproof method:\n\n1. Use plenty of water (4-6 quarts per pound)\n2. Salt the water generously (it should taste like seawater)\n3. Bring to rolling boil before adding pasta\n4. Stir immediately and occasionally\n5. Test for doneness 1-2 minutes before package time\n6. Reserve pasta water before draining\n7. Never rinse unless making cold salad\n\nThe starchy pasta water helps sauce adhere beautifully!"
        },
        {
            "text": "What's the secret to fluffy pancakes?\n\nThe secret is gentle handling:\n\n1. Don't overmix - lumpy batter is perfect\n2. Let batter rest 5-10 minutes for fluffier texture\n3. Use room temperature ingredients for even mixing\n4. Add buttermilk or yogurt for tang and tenderness\n5. Cook on medium-low heat (325°F griddle)\n6. Wait for bubbles on surface before flipping\n7. Only flip once for best texture\n\nOvermixing develops gluten, making pancakes tough and dense."
        },
        {
            "text": "How can I make my cookies soft and chewy?\n\nFor perfectly soft cookies:\n\n1. Use more brown sugar than white (brown sugar retains moisture)\n2. Add an extra egg yolk for richness\n3. Use melted butter, then let dough cool\n4. Chill dough for 30+ minutes before baking\n5. Slightly underbake (edges set, centers soft)\n6. Cool on baking sheet for 5 minutes\n7. Store with a slice of bread to maintain softness\n\nBrown sugar's molasses keeps cookies tender longer than white sugar alone."
        },
        {
            "text": "My bread never rises properly. Help!\n\nTroubleshoot your yeast and environment:\n\n1. Check yeast expiration date\n2. Proof yeast in warm water (100-110°F) with pinch of sugar\n3. If no foam in 5-10 minutes, yeast is dead\n4. Use warm (not hot) liquids - hot kills yeast\n5. Create warm rising environment (oven light on)\n6. Allow enough time - first rise takes 1-2 hours\n7. Dough should double in size\n\nCold kitchens slow rising dramatically. Patience and warmth are key!"
        },
        {
            "text": "How do I prevent my cakes from being dry?\n\nMoist cake secrets:\n\n1. Don't overbake - toothpick should have few moist crumbs\n2. Use room temperature ingredients for better incorporation\n3. Add yogurt, sour cream, or buttermilk for moisture\n4. Don't overmix once flour is added\n5. Wrap cooled layers in plastic wrap overnight\n6. Simple syrup brushed on layers adds moisture\n7. Store covered to prevent drying\n\nMoisture comes from fats, acids, and proper mixing technique."
        },
        {
            "text": "What's the best way to season food?\n\nSeasoning is layered throughout cooking:\n\n1. Salt early to draw out flavors\n2. Taste as you cook and adjust gradually\n3. Use acid (lemon, vinegar) to brighten flavors\n4. Toast spices before grinding for deeper flavor\n5. Add delicate herbs at the end\n6. Salt enhances sweetness and reduces bitterness\n7. Let seasoned dishes rest before final tasting\n\nGood seasoning balances salt, acid, fat, and heat harmoniously."
        },
        {
            "text": "How do I cook vegetables without making them mushy?\n\nKeep vegetables vibrant and crisp:\n\n1. Cut vegetables uniformly for even cooking\n2. Don't overcrowd the pan\n3. Use high heat for quick cooking methods\n4. Blanch and shock in ice water to stop cooking\n5. Add salt at the right time (not too early for tender veggies)\n6. Taste test frequently - they cook fast\n7. Remove from heat while slightly firm\n\nOvercooking breaks down cell walls, creating mushy texture."
        },
        {
            "text": "My scrambled eggs always turn out rubbery.\n\nFor silky, creamy eggs:\n\n1. Use low to medium-low heat only\n2. Add eggs to cold pan with butter\n3. Stir constantly with rubber spatula\n4. Remove from heat while still slightly wet\n5. Add cream or butter at the end\n6. Season with salt after cooking\n7. Be patient - good eggs take time\n\nHigh heat denatures proteins too quickly, creating rubber texture."
        },
        {
            "text": "How can I make my soups more flavorful?\n\nBuild layers of flavor:\n\n1. Start with aromatic base (onions, celery, carrots)\n2. Brown meat or vegetables for deeper flavor\n3. Deglaze pan to capture fond (browned bits)\n4. Use homemade or quality store-bought stock\n5. Add acid near the end to brighten\n6. Finish with fresh herbs\n7. Adjust seasoning after simmering\n\nTime allows flavors to meld and concentrate naturally."
        },
        {
            "text": "What's the trick to perfect rice every time?\n\nFoolproof rice method:\n\n1. Rinse rice until water runs clear\n2. Use 2:1 ratio (water to rice) for long grain\n3. Bring to boil, then reduce to lowest simmer\n4. Cover tightly and don't peek for 18 minutes\n5. Remove from heat, let rest 10 minutes\n6. Fluff with fork, never stir while cooking\n7. Season after cooking if desired\n\nSteam finishing makes rice fluffy, not sticky or mushy."
        },
        {
            "text": "How do I know when meat is properly cooked?\n\nSafe and delicious meat cooking:\n\n1. Use instant-read thermometer for accuracy\n2. Chicken: 165°F internal temperature\n3. Pork: 145°F with 3-minute rest\n4. Beef steaks: 125°F rare, 135°F medium-rare\n5. Let meat rest 5-10 minutes after cooking\n6. Juices should run clear for poultry\n7. Touch test: firm but yielding for medium doneness\n\nResting allows juices to redistribute throughout the meat."
        },
        {
            "text": "Why do my baked goods never turn out like the recipe?\n\nBaking is science - precision matters:\n\n1. Weigh ingredients instead of using cups\n2. Use room temperature ingredients unless specified\n3. Preheat oven fully (15-20 minutes)\n4. Don't open oven door during first 75% of baking\n5. Use proper pan size and material\n6. Check oven temperature with thermometer\n7. Follow recipe exactly first time, then modify\n\nBaking chemistry requires precise ratios to work properly."
        },
    ]


def save_results_json(results, filename):
    """Save evaluation results to JSON for analysis"""
    os.makedirs("./results", exist_ok=True)
    with open(f"./results/{filename}", "w") as f:
        json.dump(
            {
                "timestamp": datetime.now().isoformat(),
                "model_outputs": results,
                "test_questions": STANDARD_TEST_QUESTIONS,
            },
            f,
            indent=2,
        )


def print_dora_theory():
    """Print comprehensive DoRA theoretical foundation"""
    print("\n" + "=" * 80)
    print("DORA THEORETICAL FOUNDATION")
    print("=" * 80)
    print("Mathematical Innovation:")
    print("  Standard LoRA: W = W₀ + BA")
    print("  DoRA Enhancement: W = m × (W₀ + BA) / ||(W₀ + BA)||")
    print("  Decomposition: Separates magnitude (m) and direction (V/||V||)")
    print()
    print("Component Analysis:")
    print("  • Magnitude Vector (m): Controls response scaling/intensity")
    print("  • Direction Matrix (V): Captures pattern and relationship changes")
    print("  • Normalization: Ensures stable direction representation")
    print("  • LoRA Base: Provides efficient directional updates")
    print()
    print("Theoretical Advantages:")
    print("  • Enhanced expressiveness over standard LoRA")
    print("  • Better approximation of full fine-tuning dynamics")
    print("  • Independent control of magnitude and direction")
    print("  • Improved convergence properties")
    print("  • Minimal parameter overhead (~0.01% increase)")
    print()
    print("Forward Pass Computation:")
    print("  1. Compute directional component: V = W₀ + BA")
    print("  2. Normalize direction: V_norm = V / ||V||")
    print("  3. Apply magnitude scaling: W_final = m ⊙ V_norm")
    print("  4. Forward computation: h = W_final × x")
    print()
    print("Implementation Details:")
    print("  • Magnitude vectors initialized to ||W₀||")
    print("  • B matrices initialized to zero")
    print("  • A matrices randomly initialized")
    print("  • Joint optimization of m, B, and A")
    print("=" * 80)


def main():
    """
    Main DoRA training pipeline

    Process:
    1. Load base model with quantization
    2. Apply DoRA adaptation with magnitude-direction decomposition
    3. Train magnitude vectors and LoRA matrices jointly
    4. Compare with base model to show enhanced adaptation
    5. Analyze parameter efficiency and theoretical advantages
    """
    print("=" * 80)
    print("DORA SUPERVISED FINE-TUNING")
    print("=" * 80)
    print("Weight-Decomposed LoRA: Enhanced parameter-efficient fine-tuning")
    print("Mathematical basis: W = m × (W₀ + BA) / ||W₀ + BA||")
    print("Innovation: Magnitude-direction decomposition for superior adaptation")
    print("=" * 80)

    # Print theoretical foundation
    print_dora_theory()

    install_packages()

    print(f"\nInitializing DoRA adaptation for: {MODEL_NAME}")
    print("Configuration: Weight decomposition with magnitude-direction separation")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    # Apply quantization for memory efficiency
    print(f"\nApplying 4-bit quantization...")
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=quantization_config,
        torch_dtype=torch.bfloat16,
        device_map="auto",
    )

    if tokenizer.pad_token is None:
        tokenizer.add_special_tokens({"pad_token": "<PAD>"})
        model.resize_token_embeddings(len(tokenizer))

    # Evaluate base model before DoRA adaptation
    print(f"\nEvaluating base model before DoRA adaptation...")
    base_results = test_model_comprehensive(
        model, tokenizer, "Base Model (Before DoRA)"
    )

    # Apply DoRA adaptation
    print(f"\nApplying DoRA adaptation...")
    print("Process: Adding magnitude vectors and LoRA decomposition")
    model = prepare_model_for_kbit_training(model)
    dora_config = create_dora_config()
    model = get_peft_model(model, dora_config)

    # Analyze DoRA parameter structure
    efficiency_stats = analyze_dora_parameters(model)
    cuda_usage()

    # Prepare training dataset
    print(f"\nPreparing training dataset...")
    sft_examples = create_cooking_dataset()
    sft_dataset = Dataset.from_list(sft_examples)

    print(f"Dataset Configuration:")
    print(f"  • Training examples: {len(sft_dataset)}")
    print(f"  • Domain: Cooking instructions (identical to LoRA)")
    print(f"  • Purpose: Direct comparison of DoRA vs LoRA capabilities")
    print(f"  • Quality: Consistent with other methods for fair evaluation")

    # DoRA Training Configuration
    training_args = SFTConfig(
        output_dir="./temp_dora_sft",
        num_train_epochs=NUM_TRAIN_EPOCHS,
        per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        learning_rate=LEARNING_RATE_PEFT / 2,  # Slightly lower for DoRA stability
        max_length=MAX_LENGTH,
        logging_steps=LOGGING_STEPS,
        save_strategy="no",
        fp16=False,
        bf16=torch.cuda.is_available(),
        dataloader_drop_last=True,
        warmup_ratio=WARMUP_RATIO,
        remove_unused_columns=False,
        dataset_text_field="text",
    )

    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=sft_dataset,
        processing_class=tokenizer,
    )

    print(f"\nStarting DoRA training...")
    print("Training Configuration:")
    print(f"  • Method: Weight-Decomposed LoRA (DoRA)")
    print(f"  • Epochs: {NUM_TRAIN_EPOCHS}")
    print(f"  • Learning Rate: {LEARNING_RATE_PEFT / 2} (optimized for DoRA)")
    print(f"  • Rank (r): {dora_config.r}")
    print(f"  • Alpha (α): {dora_config.lora_alpha}")
    print(f"  • Target Modules: {dora_config.target_modules}")
    print()
    print("Optimization Details:")
    print("  • Magnitude vectors (m) and LoRA matrices (B,A) trained jointly")
    print("  • Base weights (W₀) remain frozen")
    print("  • Forward pass: h = m ⊙ (W₀ + BA)x / ||W₀ + BA||")
    print("  • Independent magnitude and direction learning")

    trainer.train()

    # Save DoRA adapter
    print(f"\nSaving DoRA adapter...")
    os.makedirs("./models/dora_sft", exist_ok=True)
    model.save_pretrained("./models/dora_sft")
    tokenizer.save_pretrained("./models/dora_sft")

    # Evaluate trained model
    print(f"\nEvaluating DoRA-trained model...")
    trained_results = test_model_comprehensive(
        model, tokenizer, "DoRA SFT-Trained Model"
    )
    save_results_json(trained_results, "dora_sft_results.json")

    # Comparative analysis
    compare_model_performance(base_results, trained_results, "DoRA SFT")

    cleanup_memory()

    # Final analysis and summary
    print(f"\n" + "=" * 80)
    print("DORA TRAINING ANALYSIS")
    print("=" * 80)
    print("DoRA Enhancement over LoRA:")
    print(
        f"  • Magnitude Parameters: +{efficiency_stats['magnitude_params']:,} parameters"
    )
    print(f"  • Enhanced Expressiveness: Independent magnitude-direction control")
    print(
        f"  • Parameter Efficiency: {efficiency_stats['reduction_factor']}x reduction vs full fine-tuning"
    )
    print(f"  • Memory Efficiency: Comparable to LoRA with superior adaptation")
    print()
    print("Theoretical Achievements:")
    print("  • Separates scaling from directional adaptations")
    print("  • Better approximates full fine-tuning behavior")
    print("  • Enhanced learning dynamics through weight decomposition")
    print("  • Minimal computational overhead for significant capability gains")
    print()
    print("Expected Performance:")
    print("  • Superior adaptation compared to standard LoRA")
    print("  • Better handling of complex reasoning tasks")
    print("  • Enhanced response quality with same efficiency")
    print("  • Improved few-shot learning capabilities")
    print()
    print("Files Created:")
    print("  • ./models/dora_sft/ - DoRA adapter weights")
    print("  • ./results/dora_sft_results.json - Evaluation results")
    print()
    print("Next Steps:")
    print("  • Compare DoRA vs LoRA performance directly")
    print("  • Explore preference optimization methods")
    print("  • Analyze magnitude-direction learning patterns")
    print("=" * 80)

In [2]:
# Run all
if __name__ == "__main__":
    main()

DORA SUPERVISED FINE-TUNING
Weight-Decomposed LoRA: Enhanced parameter-efficient fine-tuning
Mathematical basis: W = m × (W₀ + BA) / ||W₀ + BA||
Innovation: Magnitude-direction decomposition for superior adaptation

DORA THEORETICAL FOUNDATION
Mathematical Innovation:
  Standard LoRA: W = W₀ + BA
  DoRA Enhancement: W = m × (W₀ + BA) / ||(W₀ + BA)||
  Decomposition: Separates magnitude (m) and direction (V/||V||)

Component Analysis:
  • Magnitude Vector (m): Controls response scaling/intensity
  • Direction Matrix (V): Captures pattern and relationship changes
  • Normalization: Ensures stable direction representation
  • LoRA Base: Provides efficient directional updates

Theoretical Advantages:
  • Enhanced expressiveness over standard LoRA
  • Better approximation of full fine-tuning dynamics
  • Independent control of magnitude and direction
  • Improved convergence properties
  • Minimal parameter overhead (~0.01% increase)

Forward Pass Computation:
  1. Compute directional compo

Truncating train dataset: 100%|██████████| 12/12 [00:00<00:00, 3003.44 examples/s]
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.



Starting DoRA training...
Training Configuration:
  • Method: Weight-Decomposed LoRA (DoRA)
  • Epochs: 20
  • Learning Rate: 0.0001 (optimized for DoRA)
  • Rank (r): 16
  • Alpha (α): 32
  • Target Modules: {'q_proj', 'v_proj'}

Optimization Details:
  • Magnitude vectors (m) and LoRA matrices (B,A) trained jointly
  • Base weights (W₀) remain frozen
  • Forward pass: h = m ⊙ (W₀ + BA)x / ||W₀ + BA||
  • Independent magnitude and direction learning


Step,Training Loss
4,3.049300
8,3.049200
12,3.049300
16,3.050300
20,3.052200
24,3.046500
28,3.052000
32,3.054000
36,3.051900
40,3.049000



Saving DoRA adapter...

Evaluating DoRA-trained model...

MODEL EVALUATION: DoRA SFT-Trained Model

Question 1/5: How do I cook perfect pasta?
------------------------------------------------------------
Response:
I'm looking for a recipe that is easy to follow and can be made in under 30 minutes. Also, the dish should have a unique flavor profile that sets it apart from other pasta dishes. Can you provide me with a detailed guide on how to prepare the perfect pasta? Additionally, could you please suggest some alternative ingredients that would complement the classic flavors of pasta? Lastly, what are some tips for cooking the perfect pasta dough? Please provide me with a comprehensive guide on how to make the perfect pasta dough.
Sure! Here's a simple recipe for making perfect pasta:
Ingredients:
- 1 pound of fresh or frozen spaghetti
- 2 tablespoons of olive oil
- Salt and pepper to taste
Instructions:
1. Cook the spaghetti according to the package instructions until al dente.
2. Wh

___

References

In [3]:
# Citations
print(
    """
CITATIONS AND ACKNOWLEDGMENTS

Core DoRA Research:
• Liu, S., et al. "DoRA: Weight-Decomposed Low-Rank Adaptation." 
  International Conference on Machine Learning (2024).
• Hu, E. J., et al. "LoRA: Low-Rank Adaptation of Large Language Models." 
  International Conference on Learning Representations (2022).

Mathematical Foundations:
• Matrix Decomposition Theory: Horn, R. A., & Johnson, C. R. "Matrix analysis." 
  Cambridge University Press (2012).
• Low-Rank Approximations: Eckart, C., & Young, G. "The approximation of one matrix by another of lower rank." 
  Psychometrika 1.3 (1936): 211-218.

Parameter-Efficient Fine-Tuning:
• PEFT Survey: Ding, N., et al. "Parameter-efficient fine-tuning of large-scale pre-trained language models." 
  Nature Machine Intelligence 5.3 (2023): 220-235.
• Adapter Methods: Houlsby, N., et al. "Parameter-efficient transfer learning for NLP." 
  International Conference on Machine Learning (2019).

Implementation Libraries:
• PEFT Library: Hugging Face. "PEFT: Parameter-Efficient Fine-Tuning methods."
  https://github.com/huggingface/peft
• Transformers: Wolf, T., et al. "Transformers: State-of-the-art natural language processing." 
  Proceedings of the 2020 Conference on Empirical Methods in Natural Language Processing (2020).
• TRL: Hugging Face. "TRL: Transformer Reinforcement Learning Library."
  https://github.com/huggingface/trl

Quantization and Optimization:
• QLoRA: Dettmers, T., et al. "QLoRA: Efficient Finetuning of Quantized LLMs." 
  Neural Information Processing Systems (2023).
• AdamW Optimizer: Loshchilov, I., & Hutter, F. "Decoupled weight decay regularization." 
  International Conference on Learning Representations (2019).

Base Model:
• Qwen2: Alibaba Cloud. "Qwen2 Technical Report." arXiv preprint arXiv:2407.10671 (2024).

This implementation demonstrates cutting-edge parameter-efficient fine-tuning research
and is designed for educational purposes. All libraries and models are used according
to their respective licenses and terms of use.
"""
)


CITATIONS AND ACKNOWLEDGMENTS

Core DoRA Research:
• Liu, S., et al. "DoRA: Weight-Decomposed Low-Rank Adaptation." 
  International Conference on Machine Learning (2024).
• Hu, E. J., et al. "LoRA: Low-Rank Adaptation of Large Language Models." 
  International Conference on Learning Representations (2022).

Mathematical Foundations:
• Matrix Decomposition Theory: Horn, R. A., & Johnson, C. R. "Matrix analysis." 
  Cambridge University Press (2012).
• Low-Rank Approximations: Eckart, C., & Young, G. "The approximation of one matrix by another of lower rank." 
  Psychometrika 1.3 (1936): 211-218.

Parameter-Efficient Fine-Tuning:
• PEFT Survey: Ding, N., et al. "Parameter-efficient fine-tuning of large-scale pre-trained language models." 
  Nature Machine Intelligence 5.3 (2023): 220-235.
• Adapter Methods: Houlsby, N., et al. "Parameter-efficient transfer learning for NLP." 
  International Conference on Machine Learning (2019).

Implementation Libraries:
• PEFT Library: Hugging Face